# rStar-Math: Statistics and Probability Examples

This notebook demonstrates statistical analysis and probability problems with visualizations.

In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from src.core.mcts import MCTS
from src.core.ppm import ProcessPreferenceModel
from src.models.model_interface import ModelFactory

In [ ]:
# Initialize components
mcts = MCTS.from_config_file('config/default.json')
ppm = ProcessPreferenceModel.from_config_file('config/default.json')
model = ModelFactory.create_model('openai', os.getenv('OPENAI_API_KEY'), 'config/default.json')

## 1. Descriptive Statistics

In [ ]:
stats_problems = [
    "Find the mean, median, and mode of [2, 3, 3, 4, 4, 4, 5, 5, 6]",
    "Calculate the standard deviation of [10, 12, 15, 18, 20]",
    "Find the quartiles and IQR of [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]"
]

def visualize_distribution(data: list):
    """Visualize data distribution."""
    plt.figure(figsize=(12, 4))
    
    # Histogram
    plt.subplot(131)
    plt.hist(data, bins='auto', alpha=0.7)
    plt.title('Histogram')
    
    # Box plot
    plt.subplot(132)
    plt.boxplot(data)
    plt.title('Box Plot')
    
    # KDE plot
    plt.subplot(133)
    sns.kdeplot(data=data)
    plt.title('Density Plot')
    
    plt.tight_layout()
    plt.show()

for problem in stats_problems:
    print(f"Problem: {problem}\n")
    action, trajectory = mcts.search(problem)
    
    print("Solution Steps:")
    for step in trajectory:
        confidence = ppm.evaluate_step(step['state'], model)
        print(f"- {step['state']}")
        print(f"  Confidence: {confidence:.2f}\n")
        
    # Visualize data if present
    if '[' in problem:
        data = eval(problem[problem.find('['):problem.find(']')+1])
        visualize_distribution(data)
    print("-" * 50 + "\n")

## 2. Probability Distributions

In [ ]:
def plot_probability_distribution(dist_type: str, params: dict):
    """Plot various probability distributions."""
    plt.figure(figsize=(10, 6))
    
    if dist_type == 'normal':
        x = np.linspace(params['mu'] - 4*params['sigma'],
                       params['mu'] + 4*params['sigma'], 100)
        y = stats.norm.pdf(x, params['mu'], params['sigma'])
        plt.plot(x, y, label=f'μ={params["mu"]}, σ={params["sigma"]}')
        plt.title('Normal Distribution')
        
    elif dist_type == 'binomial':
        x = np.arange(0, params['n']+1)
        y = stats.binom.pmf(x, params['n'], params['p'])
        plt.bar(x, y, alpha=0.8, label=f'n={params["n"]}, p={params["p"]}')
        plt.title('Binomial Distribution')
        
    elif dist_type == 'poisson':
        x = np.arange(0, params['lambda']*3)
        y = stats.poisson.pmf(x, params['lambda'])
        plt.bar(x, y, alpha=0.8, label=f'λ={params["lambda"]}')
        plt.title('Poisson Distribution')
    
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

# Example distributions
distributions = [
    ('normal', {'mu': 0, 'sigma': 1}),
    ('normal', {'mu': 0, 'sigma': 2}),
    ('binomial', {'n': 20, 'p': 0.3}),
    ('poisson', {'lambda': 3})
]

for dist_type, params in distributions:
    plot_probability_distribution(dist_type, params)

## 3. Hypothesis Testing

In [ ]:
def visualize_hypothesis_test(sample1: np.ndarray, sample2: np.ndarray, test_type: str):
    """Visualize hypothesis test results."""
    plt.figure(figsize=(12, 5))
    
    # Data distribution
    plt.subplot(121)
    plt.boxplot([sample1, sample2], labels=['Sample 1', 'Sample 2'])
    plt.title('Sample Distributions')
    
    # Test results
    if test_type == 't-test':
        t_stat, p_val = stats.ttest_ind(sample1, sample2)
        test_name = "Student's t-test"
    elif test_type == 'wilcoxon':
        t_stat, p_val = stats.wilcoxon(sample1, sample2)
        test_name = "Wilcoxon signed-rank test"
    
    plt.subplot(122)
    plt.text(0.5, 0.5,
             f"Test: {test_name}\n" +
             f"Statistic: {t_stat:.4f}\n" +
             f"p-value: {p_val:.4f}\n" +
             f"Significant: {p_val < 0.05}",
             ha='center', va='center')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

# Example hypothesis tests
np.random.seed(42)
sample1 = np.random.normal(0, 1, 100)
sample2 = np.random.normal(0.5, 1, 100)

print("Comparing two samples with different means:")
visualize_hypothesis_test(sample1, sample2, 't-test')

print("\nNon-parametric test:")
visualize_hypothesis_test(sample1, sample2, 'wilcoxon')